# Deterministic onshore wind NPV

Calculate a single onshore wind NPV case using base values for technology triangular distributions, mean values for market-price scaled beta distributions, midpoint values for uniform distributions, and fixed values for deterministic parameters.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity_model import (
    calculate_capacity_kw,
    calculate_capacity_mw,
    calculate_level_cash_flow_present_value_factor,
)
from electricity_parameters import (
    ANNUAL_ELECTRICITY_OUTPUT_MWH,
    WIND_ONSHORE_CAPEX_DISTRIBUTION,
    WIND_ONSHORE_EMISSIONS,
    WIND_ONSHORE_FIXED_OPEX_DISTRIBUTION,
    WIND_ONSHORE_FUEL_CONSUMPTION,
    WIND_ONSHORE_FULL_LOAD_HOURS,
    WIND_ONSHORE_VARIABLE_OPEX_DISTRIBUTION,
    LIFETIME_ELECTRICITY_YEARS,
    RETAIL_PRICE_ELECTRICITY_EUR_PER_MWH,
)
from general_parameters import (
    CARBON_PRICE_EUR_PER_T,
    NO_FUEL_PRICE_EUR_PER_MWH_TH,
    INTEREST_RATE,
)

In [2]:
annual_output_mwh = ANNUAL_ELECTRICITY_OUTPUT_MWH.value
full_load_hours = WIND_ONSHORE_FULL_LOAD_HOURS.value
capacity_mw = calculate_capacity_mw(annual_output_mwh, full_load_hours)
capacity_kw = calculate_capacity_kw(annual_output_mwh, full_load_hours)

capex_eur_per_kw = (
    WIND_ONSHORE_CAPEX_DISTRIBUTION.lower_bound
    + WIND_ONSHORE_CAPEX_DISTRIBUTION.upper_bound
) / 2
fixed_opex_eur_per_kw_year = WIND_ONSHORE_FIXED_OPEX_DISTRIBUTION.mode
variable_opex_eur_per_mwh = WIND_ONSHORE_VARIABLE_OPEX_DISTRIBUTION.mode
fuel_consumption_mwh_th_per_mwh_e = WIND_ONSHORE_FUEL_CONSUMPTION.value
emissions_tco2_per_mwh_e = WIND_ONSHORE_EMISSIONS.value
no_fuel_price_eur_per_mwh_th = NO_FUEL_PRICE_EUR_PER_MWH_TH.value
electricity_price_eur_per_mwh = RETAIL_PRICE_ELECTRICITY_EUR_PER_MWH.value
carbon_price_eur_per_t = CARBON_PRICE_EUR_PER_T.value
lifetime_years = int(LIFETIME_ELECTRICITY_YEARS.value)
discount_rate = INTEREST_RATE.value

In [3]:
initial_capex_eur = capacity_kw * capex_eur_per_kw
annual_revenue_eur = annual_output_mwh * electricity_price_eur_per_mwh
annual_fixed_opex_eur = capacity_kw * fixed_opex_eur_per_kw_year
annual_variable_opex_eur = annual_output_mwh * variable_opex_eur_per_mwh
annual_fuel_cost_eur = (
    annual_output_mwh
    * fuel_consumption_mwh_th_per_mwh_e
    * no_fuel_price_eur_per_mwh_th
)
annual_emissions_cost_eur = (
    annual_output_mwh * emissions_tco2_per_mwh_e * carbon_price_eur_per_t
)
annual_net_cash_flow_eur = (
    annual_revenue_eur
    - annual_fixed_opex_eur
    - annual_variable_opex_eur
    - annual_fuel_cost_eur
    - annual_emissions_cost_eur
)
present_value_factor = calculate_level_cash_flow_present_value_factor(
    lifetime_years=lifetime_years,
    discount_rate=discount_rate,
)
npv_eur = -initial_capex_eur + annual_net_cash_flow_eur * present_value_factor
lifetime_output_mwh = annual_output_mwh * lifetime_years
npv_eur_per_mwh = npv_eur / lifetime_output_mwh

pd.Series(
    {
        "annual_output_mwh": annual_output_mwh,
        "capacity_mw": capacity_mw,
        "capacity_kw": capacity_kw,
        "capex_eur_per_kw": capex_eur_per_kw,
        "fixed_opex_eur_per_kw_year": fixed_opex_eur_per_kw_year,
        "variable_opex_eur_per_mwh": variable_opex_eur_per_mwh,
        "fuel_consumption_mwh_th_per_mwh_e": fuel_consumption_mwh_th_per_mwh_e,
        "no_fuel_price_eur_per_mwh_th": no_fuel_price_eur_per_mwh_th,
        "emissions_tco2_per_mwh_e": emissions_tco2_per_mwh_e,
        "electricity_price_eur_per_mwh": electricity_price_eur_per_mwh,
        "carbon_price_eur_per_t": carbon_price_eur_per_t,
        "lifetime_years": lifetime_years,
        "discount_rate": discount_rate,
        "present_value_factor": present_value_factor,
        "lifetime_output_mwh": lifetime_output_mwh,
        "initial_capex_million_eur": initial_capex_eur / 1_000_000,
        "annual_revenue_million_eur": annual_revenue_eur / 1_000_000,
        "annual_fixed_opex_million_eur": annual_fixed_opex_eur / 1_000_000,
        "annual_variable_opex_million_eur": annual_variable_opex_eur / 1_000_000,
        "annual_fuel_cost_million_eur": annual_fuel_cost_eur / 1_000_000,
        "annual_emissions_cost_million_eur": annual_emissions_cost_eur / 1_000_000,
        "annual_net_cash_flow_million_eur": annual_net_cash_flow_eur / 1_000_000,
        "npv_million_eur": npv_eur / 1_000_000,
        "npv_eur_per_mwh": npv_eur_per_mwh,
        "npv_million_eur_per_mwh": npv_eur_per_mwh / 1_000_000,
    }
)

annual_output_mwh                    1.000000e+06
capacity_mw                          4.000000e+02
capacity_kw                          4.000000e+05
capex_eur_per_kw                     1.600000e+03
fixed_opex_eur_per_kw_year           3.200000e+01
variable_opex_eur_per_mwh            7.000000e+00
fuel_consumption_mwh_th_per_mwh_e    0.000000e+00
no_fuel_price_eur_per_mwh_th         0.000000e+00
emissions_tco2_per_mwh_e             0.000000e+00
electricity_price_eur_per_mwh        9.407000e+01
carbon_price_eur_per_t               8.000000e+01
lifetime_years                       2.500000e+01
discount_rate                        8.000000e-02
present_value_factor                 1.067478e+01
lifetime_output_mwh                  2.500000e+07
initial_capex_million_eur            6.400000e+02
annual_revenue_million_eur           9.407000e+01
annual_fixed_opex_million_eur        1.280000e+01
annual_variable_opex_million_eur     7.000000e+00
annual_fuel_cost_million_eur         0.000000e+00
